# Create & Plot Embeddings\nUses `SignalDataloader` + `on_after_batch_transfer` exactly as in `gwak/train/plots.py`,\nthen plots the processed (whitened) strain and the corresponding embedding for each sample.

In [ ]:
# ── Paths derived from Snakefile (make_plots_i / make_plots rules) ────────────
import os, sys
from pathlib import Path

# Same env vars as the Snakefile
GWAK_DIR    = Path(os.environ.get('GWAK_DIR',    '/Users/katya/gwak'))
OUTPUT_DIR  = Path(os.environ.get('GWAK_OUTPUT_DIR', GWAK_DIR / 'gwak/output'))

# Wildcards used in the make_plots rule
CL_CONFIG  = 'ResNet'
FM_CONFIG  = 'NF_from_file_conditioning'
IFOS       = 'HL'

# Paths exactly as make_plots_i constructs them
EMBEDDING_MODEL = OUTPUT_DIR / f'{CL_CONFIG}_{IFOS}/model_JIT.pt'
FM_MODEL        = OUTPUT_DIR / f'{CL_CONFIG}_{FM_CONFIG}_{IFOS}/model_JIT.pt'
DATA_DIR        = OUTPUT_DIR / f'BBC_AnalysisReady_Cat12/{IFOS}/'
CONFIG_PATH     = GWAK_DIR   / f'gwak/train/configs/{CL_CONFIG}.yaml'

sys.path.insert(0, str(GWAK_DIR))
sys.path.insert(0, str(GWAK_DIR / 'gwak/train'))

N_PLOT = 6   # number of samples to visualise

for label, p in [('embedding_model', EMBEDDING_MODEL),
                 ('fm_model',         FM_MODEL),
                 ('data_dir',         DATA_DIR),
                 ('config',           CONFIG_PATH)]:
    status = '✓' if Path(p).exists() else '✗ NOT FOUND'
    print(f'{status}  {label}: {p}')

## 1. Load model & config (same as plots.py)

In [ ]:
import yaml
import numpy as np
import torch
import matplotlib.pyplot as plt

from ml4gw.distributions import PowerLaw
from ml4gw.waveforms import SineGaussian, MultiSineGaussian, IMRPhenomPv2, Gaussian, GenerateString, WhiteNoiseBurst
from gwak.train.dataloader import SignalDataloader
from gwak.data.prior import (
    SineGaussianBBC, MultiSineGaussianBBC, LAL_BBHPrior,
    GaussianBBC, CuspBBC, KinkBBC, KinkkinkBBC, WhiteNoiseBurstBBC
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

embed_model = torch.jit.load(EMBEDDING_MODEL, map_location=device)
embed_model.eval()
print('Model loaded')

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

cfg              = config['data']['init_args']
sample_rate      = cfg['sample_rate']
kernel_length    = cfg['kernel_length']
psd_length       = cfg['psd_length']
fduration        = cfg['fduration']
fftlength        = cfg['fftlength']
num_workers      = cfg['num_workers']
data_saving_file = cfg.get('data_saving_file', None)
duration         = fduration + kernel_length
batch_size       = 32   # small for notebook

print(f'sample_rate={sample_rate}, kernel_length={kernel_length}s, psd_length={psd_length}s')

## 2. Set up SignalDataloader (same as plots.py)

In [ ]:
signal_classes = [
    'MultiSineGaussian', 'SineGaussian', 'BBH', 'Gaussian',
    'Cusp', 'Kink', 'KinkKink', 'WhiteNoiseBurst',
    'CCSN', 'Background', 'Glitch',
]

priors = [
    MultiSineGaussianBBC(), SineGaussianBBC(), LAL_BBHPrior(), GaussianBBC(),
    CuspBBC(), KinkBBC(), KinkkinkBBC(), WhiteNoiseBurstBBC(),
    None, None, None,
]

waveforms = [
    MultiSineGaussian(sample_rate=sample_rate, duration=duration),
    SineGaussian(sample_rate=sample_rate, duration=duration),
    IMRPhenomPv2(),
    Gaussian(sample_rate=sample_rate, duration=duration),
    GenerateString(sample_rate=sample_rate),
    GenerateString(sample_rate=sample_rate),
    GenerateString(sample_rate=sample_rate),
    WhiteNoiseBurst(sample_rate=sample_rate, duration=duration),
    None, None, None,
]

extra_kwargs = [
    None, None, {'ringdown_duration': 0.9}, None,
    None, None, None, None,
    None, None, None,
]

loader = SignalDataloader(
    signal_classes, priors, waveforms, extra_kwargs,
    data_dir=DATA_DIR,
    sample_rate=sample_rate,
    kernel_length=kernel_length,
    psd_length=psd_length,
    fduration=fduration,
    fftlength=fftlength,
    batch_size=batch_size,
    batches_per_epoch=100,
    num_workers=num_workers,
    data_saving_file=data_saving_file,
    ifos=IFOS,
    snr_prior=PowerLaw(index=3, minimum=4, maximum=30),
)

print('Dataloader ready')

## 3. Get one batch & preprocess (same as plots.py)

In [ ]:
test_loader = loader.test_dataloader()
test_iter   = iter(test_loader)

clean_batch, glitch_batch = next(test_iter)
clean_batch  = clean_batch.to(device)
glitch_batch = glitch_batch.to(device)

# on_after_batch_transfer does whitening + bandpass, identical to training
processed, labels, snrs, hrss = loader.on_after_batch_transfer(
    [clean_batch, glitch_batch], None, local_test=True
)

print('processed shape:', processed.shape)   # [batch, num_ifos, kernel_samples]
print('labels shape:',    labels.shape)
print('unique labels:',   labels.unique().cpu().numpy())

## 4. Compute embeddings

In [ ]:
with torch.no_grad():
    embeddings = embed_model(processed)   # [batch, embed_dim]

processed_np  = processed.detach().cpu().numpy()   # [B, C, T]
embeddings_np = embeddings.detach().cpu().numpy()  # [B, D]
labels_np     = labels.detach().cpu().numpy()
snrs_np       = snrs.detach().cpu().numpy()

print('embeddings shape:', embeddings_np.shape)

## 5. Plot: whitened strain → embedding (per sample)

In [ ]:
ifos   = [f'{c}1' for c in IFOS]       # e.g. ['H1', 'L1']
t_axis = np.linspace(0, kernel_length, processed_np.shape[-1])

for i in range(min(N_PLOT, len(processed_np))):
    lbl  = int(labels_np[i])
    name = signal_classes[lbl - 1] if 1 <= lbl <= len(signal_classes) else f'label {lbl}'
    snr  = float(snrs_np[i]) if snrs_np.ndim == 1 else float(snrs_np[i].item())

    n_rows = len(ifos) + 1   # one row per IFO + embedding bar
    fig, axes = plt.subplots(n_rows, 1, figsize=(13, 2.8 * n_rows))
    fig.suptitle(f'Sample {i} — class: {name}  SNR: {snr:.1f}', fontsize=12)

    for ch, ifo in enumerate(ifos):
        axes[ch].plot(t_axis, processed_np[i, ch], linewidth=0.6)
        axes[ch].set_ylabel(f'{ifo} whitened')
        axes[ch].set_xlabel('time (s)')

    emb = embeddings_np[i]
    axes[-1].bar(range(len(emb)), emb, color='mediumseagreen')
    axes[-1].set_xlabel('embedding dimension')
    axes[-1].set_ylabel('value')
    axes[-1].set_title('embedding output')

    plt.tight_layout()
    plt.show()

## 6. Embedding dims over sample index (full batch)

In [ ]:
n_dims_show = min(6, embeddings_np.shape[1])

all_background = ['Background', 'Glitch']
bg_labels = [i + 1 for i, c in enumerate(signal_classes) if c in all_background]
is_signal = ~np.isin(labels_np, bg_labels)

fig, ax = plt.subplots(figsize=(13, 5))
for d in range(n_dims_show):
    ax.plot(embeddings_np[:, d], linewidth=0.8, label=f'dim {d}')

# Shade signal samples in red
for idx in np.where(is_signal)[0]:
    ax.axvspan(idx - 0.5, idx + 0.5, color='red', alpha=0.15)

ax.set_xlabel('sample index')
ax.set_ylabel('embedding dims readout')
ax.set_title('Embedding dims over batch  (red shading = signal sample)')
ax.legend()
plt.tight_layout()
plt.show()